In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model, Model

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs Detected: {len(gpus)}")
if len(gpus) == 0:
    raise ValueError("No GPU found!")

tf.keras.mixed_precision.set_global_policy('mixed_float16')

BILSTM_PATH = '/kaggle/working/bilstm_cholec80.keras'
I3D_PATH = '/kaggle/input/models/ganuwoahh/140426-final-keras/pytorch/default/1/140426_final.keras' 
FEATURE_PATH = '/kaggle/input/datasets/ganuwoahh/cholec80-i3d-features-real/video72_features.npy' 
RGB_CLIP_DIR = '/kaggle/input/datasets/ganuwoahh/temp72/video72' # just importing one video from the validation set since we didn't have a test set


# so again how you find the below information is up to you. i haven't implemented a front end or inference system so it's manual right now
TARGET_TIMESTEP = 300 # exact second you want to analyse
TARGET_CLASS = 3      # what phase you want to analyse for

strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
    bilstm = load_model(BILSTM_PATH)

    full_i3d = load_model(I3D_PATH)

last_conv_layer_name = None
for layer in full_i3d.layers: # we need the last convolution layer before pooling
    if 'Mixed_5c_rgb' in layer.name or 'mixed_5c_rgb' in layer.name:
        last_conv_layer_name = layer.name
        break

if last_conv_layer_name is None:
    conv_layers = [l for l in full_i3d.layers if isinstance(l, tf.keras.layers.Conv3D) and 'rgb' in l.name]
    last_conv_layer_name = conv_layers[-1].name

spatial_i3d = Model(inputs=full_i3d.input, outputs=full_i3d.get_layer(last_conv_layer_name).output)

features = np.load(FEATURE_PATH)
features_tensor = tf.convert_to_tensor(np.expand_dims(features, axis=0), dtype=tf.float32)

with tf.GradientTape() as tape:
    tape.watch(features_tensor)
    preds = bilstm(features_tensor)
    target_score = preds[0, TARGET_TIMESTEP, TARGET_CLASS]

bilstm_grads = tape.gradient(target_score, features_tensor)[0, TARGET_TIMESTEP, :].numpy()

start_frame = TARGET_TIMESTEP * 8
frames = sorted(os.listdir(RGB_CLIP_DIR))[start_frame : start_frame + 32]

clip_rgb = []
for f_name in frames:
    img = cv2.imread(os.path.join(RGB_CLIP_DIR, f_name))
    img = cv2.resize(img, (224, 224))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    clip_rgb.append(img)

clip_tensor_rgb = np.array(clip_rgb, dtype=np.float32) / 127.5 - 1.0
clip_tensor_rgb = np.expand_dims(clip_tensor_rgb, axis=0)

dummy_flow = np.zeros((1, 32, 224, 224, 2), dtype=np.float32)

# using strategy.scope() because not using it upset keras for some reason
with strategy.scope():
    conv_outputs = spatial_i3d.predict([clip_tensor_rgb, dummy_flow])[0] 

spatial_map = np.mean(conv_outputs, axis=0) 

for i in range(spatial_map.shape[-1]):
    spatial_map[:, :, i] *= bilstm_grads[i]

heatmap = np.sum(spatial_map, axis=-1)
heatmap = np.maximum(heatmap, 0)
heatmap /= np.max(heatmap) + 1e-10 

heatmap = np.float32(heatmap)

center_img = clip_rgb[16]
heatmap_resized = cv2.resize(heatmap, (center_img.shape[1], center_img.shape[0]))
heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)

superimposed_img = cv2.addWeighted(center_img, 0.6, heatmap_colored, 0.4, 0)

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.title(f"Original Frame")
plt.imshow(center_img)
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title("Grad-CAM++ Spatial Activation")
plt.imshow(heatmap, cmap='jet')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title(f"Overlay")
plt.imshow(superimposed_img)
plt.axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/bilstm_gradcam.png', dpi=300)
plt.show()